<a href="https://colab.research.google.com/github/Adhiaris/UAS-FINALTERM-FOR-DEEP-LEARNING-Question-Answering/blob/main/Task_2_Question_Answering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 2: Question Answering with T5-base on SQuAD
### NLP Assignment — Encoder-Decoder (Seq2Seq) Architecture

## Step 1: Install Dependencies

In [ ]:
!pip install transformers datasets evaluate accelerate sentencepiece rouge_score -q
print(" All packages installed!")

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import re
from collections import Counter

from datasets import load_dataset
from transformers import (
 AutoTokenizer,
 AutoModelForSeq2SeqLM,
 DataCollatorForSeq2Seq,
 Seq2SeqTrainingArguments,
 Seq2SeqTrainer,
 EarlyStoppingCallback
)
import evaluate

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Device : {device}")
if torch.cuda.is_available():
 print(f" GPU : {torch.cuda.get_device_name(0)}")
 print(f" VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
 print(" No GPU — training will be very slow. Please enable GPU.")
print(f" PyTorch: {torch.__version__}")

## Step 2: Load & Explore SQuAD

In [ ]:
dataset = load_dataset('rajpurkar/squad')
print(dataset)
print("\n Sample entry:")
sample = dataset['train'][0]
for k, v in sample.items():
 print(f" {k}: {v}")

In [ ]:
print(f"Train: {len(dataset['train']):,} examples")
print(f"Val : {len(dataset['validation']):,} examples")

train_sample = dataset['train'].select(range(5000))
answer_lens = [len(ex['answers']['text'][0].split()) for ex in train_sample]
context_lens = [len(ex['context'].split()) for ex in train_sample]
question_lens= [len(ex['question'].split()) for ex in train_sample]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(answer_lens, bins=40, color='#2ecc71', alpha=0.85, edgecolor='white')
axes[0].set_title('Answer Length (words)', fontweight='bold')
axes[0].set_xlabel('Words')
axes[0].axvline(np.percentile(answer_lens, 95), color='red',
 linestyle='--', label=f'95th pct={np.percentile(answer_lens,95):.0f}')
axes[0].legend()

axes[1].hist(context_lens, bins=40, color='#3498db', alpha=0.85, edgecolor='white')
axes[1].set_title('Context Length (words)', fontweight='bold')
axes[1].set_xlabel('Words')
axes[1].axvline(400, color='red', linestyle='--', label='400w cutoff')
axes[1].legend()

axes[2].hist(question_lens, bins=30, color='#e67e22', alpha=0.85, edgecolor='white')
axes[2].set_title('Question Length (words)', fontweight='bold')
axes[2].set_xlabel('Words')

plt.suptitle('SQuAD Dataset EDA', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('squad_eda.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Answer length — mean: {np.mean(answer_lens):.1f} | 95th pct: {np.percentile(answer_lens,95):.0f}")
print(f"Context length — mean: {np.mean(context_lens):.1f}")
print(f"Question length — mean: {np.mean(question_lens):.1f}")

In [ ]:
print(" Sample QA Pairs:\n")
for i in range(3):
 ex = dataset['train'][i]
 print(f"[{i+1}] Context : {ex['context'][:120]}...")
 print(f" Question : {ex['question']}")
 print(f" Answer : {ex['answers']['text'][0]}")
 print()

## Step 3: Tokenization & Preprocessing

In [ ]:
MODEL_NAME = 't5-base'
MAX_INPUT = 512
MAX_TARGET = 64

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f" Tokenizer: {MODEL_NAME} | Vocab: {tokenizer.vocab_size:,}")

demo_input = "question: What is the capital of France? context: France is a country in Europe. Its capital is Paris."
demo_tokens = tokenizer(demo_input, return_tensors='pt')
print(f"\n T5 input format example:")
print(f" Input : {demo_input}")
print(f" Tokens : {demo_tokens['input_ids'].shape[1]} tokens")

In [ ]:
def preprocess_squad(examples):
 inputs = [f"question: {q} context: {c}"
 for q, c in zip(examples['question'], examples['context'])]
 targets = [ans['text'][0] if ans['text'] else ''
 for ans in examples['answers']]

 model_inputs = tokenizer(
 inputs,
 text_target=targets,
 max_length=MAX_INPUT,
 max_target_length=MAX_TARGET,
 truncation=True,
 padding=False
 )
 return model_inputs

tokenized = dataset.map(
 preprocess_squad, batched=True,
 remove_columns=dataset['train'].column_names
)
print("Preprocessing complete!")
print(tokenized)

In [ ]:
USE_SUBSET = True
TRAIN_SIZE = 15000
EVAL_SIZE = 1000

if USE_SUBSET:
 train_ds = tokenized['train'].shuffle(seed=42).select(range(TRAIN_SIZE))
 eval_ds = tokenized['validation'].shuffle(seed=42).select(range(EVAL_SIZE))
else:
 train_ds = tokenized['train']
 eval_ds = tokenized['validation']

print(f" Train: {len(train_ds):,} | Eval: {len(eval_ds):,}")

## Step 4: Load T5-base Model

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f" Model: {MODEL_NAME}")
print(f" Total parameters : {total_params:,} (~{total_params/1e6:.0f}M)")
print(f" Trainable parameters: {trainable_params:,}")
print()
print("Architecture:")
print(" 'question: Q context: C' → [T5 Encoder] → [T5 Decoder] → 'answer text'")
print(" T5 = Text-to-Text Transfer Transformer (Raffel et al., 2020)")

## Step 5: Training

In [ ]:
def normalize_answer(s):
    """Lowercase, remove punctuation/articles/extra whitespace."""
    s = s.lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = re.sub(r'[^\w\s]', '', s)
    return ' '.join(s.split())

def compute_em_f1(pred_str, gold_str):
    pred_tokens = normalize_answer(pred_str).split()
    gold_tokens = normalize_answer(gold_str).split()
    em = int(normalize_answer(pred_str) == normalize_answer(gold_str))
    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return em, 0.0

    precision = num_same / len(pred_tokens)
    recall    = num_same / len(gold_tokens)
    f1 = 2 * precision * recall / (precision + recall)
    return em, f1

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]
    preds  = np.where(preds  != -100, preds,  tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds  = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]
    em_scores, f1_scores = [], []
    for pred, label in zip(decoded_preds, decoded_labels):
        em, f1 = compute_em_f1(pred, label)
        em_scores.append(em)
        f1_scores.append(f1)
    return {
        'exact_match': np.mean(em_scores),
        'f1':          np.mean(f1_scores)
    }

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=model, padding=True
)

training_args = Seq2SeqTrainingArguments(
    output_dir='/content/results/t5-squad',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=3e-4,
    weight_decay=0.01,
    warmup_steps=200,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=100,
    report_to='none',
    seed=42,
    fp16=torch.cuda.is_available()
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)
print("Seq2SeqTrainer ready. Starting training...")

In [ ]:
train_result = trainer.train()
print("\n Training complete!")
print(f" Loss : {train_result.training_loss:.4f}")
print(f" Runtime: {train_result.metrics['train_runtime']:.0f}s")

## Step 6: Evaluation

In [ ]:
eval_results = trainer.evaluate(eval_ds)
print(" T5 QA Evaluation Results:")
for k, v in eval_results.items():
  if isinstance(v, float):
      print(f" {k}: {v:.4f}")

In [ ]:
sample_eval = dataset['validation'].select(range(200))

questions = [ex['question'] for ex in sample_eval]
contexts = [ex['context'] for ex in sample_eval]
gold_answers = [ex['answers']['text'][0] for ex in sample_eval]

def generate_answer(question, context):
 input_text = f"question: {question} context: {context}"
 inputs = tokenizer(input_text, return_tensors='pt',
 max_length=MAX_INPUT, truncation=True).to(device)
 with torch.no_grad():
  outputs = model.generate(
    **inputs,
    max_length=inputs['input_ids'].shape[1] + MAX_TARGET,
    num_beams=4,
    early_stopping=True
)
 return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("⏳ Generating predictions on 200 validation samples...")
preds = [generate_answer(q, c) for q, c in zip(questions, contexts)]

em_list, f1_list = [], []
for pred, gold in zip(preds, gold_answers):
 em, f1 = compute_em_f1(pred, gold)
 em_list.append(em)
 f1_list.append(f1)

print(f"\n Results on 200 samples:")
print(f" Exact Match: {np.mean(em_list)*100:.1f}%")
print(f" Token F1 : {np.mean(f1_list)*100:.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].hist(f1_list, bins=20, color='#3498db', alpha=0.85, edgecolor='white')
axes[0].axvline(np.mean(f1_list), color='red', linestyle='--',
 label=f'Mean F1={np.mean(f1_list):.3f}')
axes[0].set_title('Token F1 Score Distribution', fontweight='bold')
axes[0].set_xlabel('F1 Score')
axes[0].set_ylabel('Count')
axes[0].legend()

em_count = sum(em_list)
axes[1].pie([em_count, len(em_list)-em_count],
 labels=['Exact Match', 'Partial/No Match'],
 colors=['#2ecc71','#e74c3c'],
 autopct='%1.1f%%', startangle=90,
 wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title(f'Exact Match Rate\n({em_count}/{len(em_list)} samples)', fontweight='bold')

logs = trainer.state.log_history
train_loss = [(e['epoch'], e['loss']) for e in logs if 'loss' in e]
eval_f1 = [(e['epoch'], e['eval_f1']) for e in logs if 'eval_f1' in e]
eval_loss = [(e['epoch'], e['eval_loss']) for e in logs if 'eval_loss' in e]
if train_loss:
 axes[2].plot(*zip(*train_loss), 'o-', label='Train Loss', markersize=3)
if eval_loss:
 axes[2].plot(*zip(*eval_loss), 's-', label='Eval Loss', markersize=5)
ax2b = axes[2].twinx()
if eval_f1:
 ax2b.plot(*zip(*eval_f1), 'D--', color='green', label='Eval F1', markersize=6)
 ax2b.set_ylabel('F1 Score', color='green')
 ax2b.tick_params(axis='y', labelcolor='green')
 ax2b.legend(loc='upper right')
axes[2].set_title('Training Curves', fontweight='bold')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Loss')
axes[2].legend(loc='upper left')

plt.suptitle('T5-base QA Results on SQuAD', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('t5_squad_results.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 7: Inference Demo

In [ ]:
def qa(question, context, num_beams=4):
    input_text = f"question: {question}  context: {context}"
    inputs = tokenizer(input_text, return_tensors='pt',
                       max_length=MAX_INPUT, truncation=True).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=inputs['input_ids'].shape[1] + MAX_TARGET,
            num_beams=num_beams,
            early_stopping=True
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(" QA Inference Demo:")
print("=" * 70)
for i in range(5):
 ex = dataset['validation'][i]
 pred = qa(ex['question'], ex['context'])
 gold = ex['answers']['text'][0]
 em, f1 = compute_em_f1(pred, gold)
 match = "" if em else ("🟡" if f1 > 0.5 else "")
 print(f"[{i+1}] Question : {ex['question']}")
 print(f" Gold : {gold}")
 print(f" Predicted: {pred} {match} (F1={f1:.2f})")
 print()

In [ ]:
custom_context = """
The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France.
It is named after the engineer Gustave Eiffel, whose company designed and built the tower
from 1887 to 1889 as the entrance arch for the 1889 World's Fair.
The tower is 330 metres tall and was the world's tallest man-made structure for 41 years.
"""

custom_questions = [
 "Who designed the Eiffel Tower?",
 "When was the Eiffel Tower built?",
 "How tall is the Eiffel Tower?",
 "What was the Eiffel Tower built for?",
]

print(" Custom Context QA:")
print("=" * 65)
for q in custom_questions:
 answer = qa(q, custom_context)
 print(f"Q: {q}")
 print(f"A: {answer}")
 print()

## Step 8: Save Model & Metrics

In [ ]:
SAVE_PATH = '/content/saved_models/t5-squad'
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

metrics_summary = {
 'model': MODEL_NAME,
 'dataset': 'SQuAD v1.1',
 'task': 'Generative Question Answering',
 'train_samples': len(train_ds),
 'eval_samples': len(eval_ds),
 'eval_exact_match': eval_results.get('eval_exact_match'),
 'eval_f1': eval_results.get('eval_f1'),
 'eval_loss': eval_results.get('eval_loss'),
 'hyperparams': {
 'epochs': 3, 'lr': 3e-4, 'batch_size': 16,
 'max_input': MAX_INPUT, 'max_target': MAX_TARGET,
 'num_beams': 4
 }
}
with open('t5_squad_metrics.json', 'w') as f:
 json.dump(metrics_summary, f, indent=2)

print(f" Model saved → {SAVE_PATH}")
print(f" Metrics saved → t5_squad_metrics.json")